# Week 1 Day 1 - my exercises

1. Suggest a subject line for an email
2. Summarize Javascript-rendered websites (like openai.com) using a Playwright version of the scraper

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
import scraper_playwright

load_dotenv(override=True)
openai = OpenAI()

## 1. Email subject line suggester

In [ ]:
# Step 1: Create your prompts

email_system_prompt = """
You are an assistant that reads the contents of an email and suggests a short, clear subject line for it.
The subject line should be at most 8 words and capture the main purpose of the email.
Reply in the same language as the email. Respond with only the subject line - no quotes, no explanation.
"""

email_user_prompt = """
Halo Pak Budi,

Saya ingin mengonfirmasi bahwa rapat evaluasi proyek aplikasi mobile yang semula dijadwalkan hari Kamis
dipindah ke hari Jumat, 17 Oktober, pukul 14.00 di ruang meeting lantai 3.
Mohon siapkan laporan progres sprint terakhir dan daftar bug yang masih terbuka.

Terima kasih,
Rina
"""

# Step 2: Make the messages list

email_messages = [
    {"role": "system", "content": email_system_prompt},
    {"role": "user", "content": email_user_prompt}
]

# Step 3: Call OpenAI
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=email_messages)

# Step 4: print the result
print(response.choices[0].message.content)

## 2. Summarizing Javascript websites with Playwright

The simple `requests` + BeautifulSoup scraper only gets a placeholder from openai.com ("Enable JavaScript and cookies to continue").

`scraper_playwright.py` has the same functions as `week1/scraper.py`, but opens the page in a headless Edge browser, waits for the Javascript and the Cloudflare check, and then reads the text.

One-time install: `uv pip install playwright` (no `playwright install` needed, as it uses the Edge already on Windows).

In [ ]:
print(scraper_playwright.fetch_website_contents("https://openai.com")[:500])

In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def display_summary_js(url):
    website = scraper_playwright.fetch_website_contents(url)
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages_for(website))
    display(Markdown(response.choices[0].message.content))

In [ ]:
display_summary_js("https://openai.com")